In [0]:
# Databricks Notebook: Loan_Branch_Silver
# Cell 1: Clean and Merge Branch & Loan Data

from pyspark.sql.functions import col, current_timestamp, to_date, trim, upper
from delta.tables import DeltaTable

# 1. Branch Cleansing (Dimension Standardization)
df_clean_branch = spark.table("bankingpoc.bronze.branch").select(
    col("branch_id").cast("int"),
    trim(col("branch_name")).alias("branch_name"),
    trim(col("city")).alias("city"),
    upper(trim(col("ifsc_code"))).alias("ifsc_code"),
    trim(col("branch_type")).alias("branch_type"),
    current_timestamp().alias("silver_processed_timestamp")
)

df_clean_branch.write.format("delta").mode("overwrite").saveAsTable("bankingpoc.silver.branch")
print("Branch table updated in Silver.")

# 2. Loan Cleansing & Consistency Validation
df_clean_loan = spark.table("bankingpoc.bronze.loan").select(
    col("loan_id").cast("int"),
    col("customer_id").cast("int"),
    trim(col("loan_type")).alias("loan_type"),
    col("loan_amount").cast("decimal(18,2)"),
    col("paid_amount").cast("decimal(18,2)"),
    (col("paid_amount") <= col("loan_amount")).alias("is_payment_consistent"),
    trim(col("loan_status")).alias("loan_status"),
    to_date(col("start_date")).alias("start_date"),
    current_timestamp().alias("silver_processed_timestamp")
)

target_loan = "bankingpoc.silver.loan"

if not spark.catalog.tableExists(target_loan):
    df_clean_loan.write.format("delta").mode("overwrite").saveAsTable(target_loan)
    print(f"Initialized table: {target_loan}")
else:
    DeltaTable.forName(spark, target_loan).alias("tgt").merge(
        source=df_clean_loan.alias("src"),
        condition="tgt.loan_id = src.loan_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("Loan upsert completed successfully.")

Branch table updated in Silver.
Initialized table: bankingpoc.silver.loan
